# 11_行銷_活動成效

比率與長條圖

任務：逐列計算點擊率與每筆名單成本。；分母為 0 時保留空白，不能顯示為 0 元。；畫各管道名單數長條圖，算整體每筆名單成本。

規則：點擊率＝點擊數÷曝光數；每筆名單成本＝花費÷名單數；整體比率用總和相除。

先人工核對再執行。資料皆為教學模擬值。

In [ ]:
%pip -q install openpyxl numpy matplotlib

In [ ]:
from pathlib import Path
import csv
from openpyxl import load_workbook

# 本機：把 Notebook 與 Excel 放在同一資料夾。Colab：先上傳本例 Excel。
SOURCE = Path('11_行銷_活動成效.xlsx')
if not SOURCE.exists():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f'請將 {SOURCE.name} 放到目前工作目錄：{Path.cwd()}')
    files.upload()
if not SOURCE.exists():
    raise FileNotFoundError(f'尚未找到 {SOURCE.name}，請確認上傳檔名')
OUT = Path('output') / '11_行銷_活動成效'
OUT.mkdir(parents=True, exist_ok=True)

def read_rows(sheet):
    data = sheet.iter_rows(values_only=True)
    headers = next(data)
    return [dict(zip(headers, row)) for row in data if any(v is not None for v in row)]

def save_csv(name, records):
    if not records:
        print('沒有符合條件的資料，未輸出 CSV')
        return
    with (OUT / name).open('w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)

# 來源檔只有原始值，沒有需要重算的 Excel 公式。
wb = load_workbook(SOURCE, read_only=True, data_only=True)
print('工作表：', wb.sheetnames)
rows = read_rows(wb['資料'])
print('第一筆：', rows[0])

In [ ]:
import matplotlib.pyplot as plt
result = [dict(r, 點擊率=r['點擊數']/r['曝光數'] if r['曝光數'] else None,
               每筆名單成本=r['花費_元']/r['名單數'] if r['名單數'] else None) for r in rows]
overall = sum(r['花費_元'] for r in rows) / sum(r['名單數'] for r in rows)
print(result, '整體每筆名單成本：', overall)
save_csv('活動成效.csv', result)
fig, ax = plt.subplots()
ax.bar([r['活動編號'] for r in rows], [r['名單數'] for r in rows])
ax.set(title='Campaign leads', xlabel='Campaign ID', ylabel='Leads')
fig.tight_layout()
fig.savefig(OUT / '活動名單數.png', dpi=150)
plt.close(fig)
assert result[3]['每筆名單成本'] is None
assert overall == 26000/270
wb.close()

## 驗收答案

M001、M002、M003 名單成本為 100、100、50 元；M004 空白；整體約 96.30 元。